# Create Genie Space for Logistics Control Center

This notebook creates or updates a Databricks Genie Space that enables natural language queries over logistics metric views.

**What is Genie Space?**
Genie Space is a Databricks feature that allows users to ask questions in natural language and get SQL-powered answers from metric views.

**Metric Views Used:**
- `network_metrics` - Overall network health and performance
- `shipment_metrics` - Shipment volume and status metrics
- `incident_metrics` - Incident frequency and impact metrics
- `capacity_metrics` - Lane capacity utilization metrics

## Configuration

Load catalog and schema from job parameters, environment variables, or use defaults.

In [ ]:
from __future__ import annotations

import json
import os


def _get_config(name: str, default: str) -> str:
    """Get config from job parameters (widgets), env vars, or default."""
    try:
        return dbutils.widgets.get(name)  # type: ignore[name-defined]
    except Exception:
        return os.environ.get(f"DATABRICKS_{name.upper()}", default)


CATALOG = _get_config("catalog", "main")
SCHEMA = _get_config("schema", "logistics_control_center")
DISPLAY_NAME = "Logistics Control Center Metrics"
DESCRIPTION = "Natural language analytics over logistics control center metric views."

try:
    WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")  # type: ignore[name-defined]
except Exception:
    WAREHOUSE_ID = ""

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Genie Space Name: {DISPLAY_NAME}")

## Define Metric Views

List of Unity Catalog metric views to include in the Genie Space. These views must exist before creating the space.

In [ ]:
METRIC_VIEWS = [
    f"{CATALOG}.{SCHEMA}.network_metrics",
    f"{CATALOG}.{SCHEMA}.shipment_metrics",
    f"{CATALOG}.{SCHEMA}.incident_metrics",
    f"{CATALOG}.{SCHEMA}.capacity_metrics",
]

print("Metric views to include:")
for mv in METRIC_VIEWS:
    print(f"  - {mv}")

## Initialize Databricks SDK Client

Connect to the workspace using the notebook's authentication context.

In [ ]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient()
print(f"Connected to workspace: {client.config.host}")

## Resolve SQL Warehouse

Find a SQL warehouse to power the Genie Space queries. Uses the provided warehouse ID or finds the first available one.

In [ ]:
def _resolve_warehouse(client: WorkspaceClient, candidate_id: str) -> str:
    """Find a valid SQL warehouse ID."""
    if candidate_id:
        return candidate_id
    for wh in client.warehouses.list():
        if wh.id:
            return wh.id
    raise RuntimeError("No SQL warehouse found for Genie space creation.")


warehouse_id = _resolve_warehouse(client, WAREHOUSE_ID)
print(f"Using SQL Warehouse: {warehouse_id}")

## Build Genie Space Payload

Create the serialized space configuration that defines which metric views to include.

In [ ]:
def _serialized_space() -> str:
    """Build the JSON payload for Genie Space configuration."""
    payload = {
        "version": 2,
        "data_sources": {
            "metric_views": [{"identifier": mv} for mv in sorted(METRIC_VIEWS)]
        },
    }
    return json.dumps(payload)


serialized = _serialized_space()
print("Genie Space configuration:")
print(json.dumps(json.loads(serialized), indent=2))

## Check for Existing Genie Space

Look for an existing Genie Space with the same name. If found, we'll update it; otherwise, we'll create a new one.

In [ ]:
existing = None
resp = client.genie.list_spaces()

for space in resp.spaces or []:
    if getattr(space, "title", None) == DISPLAY_NAME:
        existing = space
        break

if existing:
    print(f"Found existing Genie Space: {existing.space_id}")
else:
    print("No existing Genie Space found. Will create a new one.")

## Create or Update Genie Space

Create a new Genie Space or update the existing one with the current configuration.

In [ ]:
if existing:
    # Update existing space
    if hasattr(client.genie, "update_space"):
        updated = client.genie.update_space(
            space_id=existing.space_id,
            title=DISPLAY_NAME,
            description=DESCRIPTION,
            warehouse_id=warehouse_id,
            serialized_space=serialized,
        )
        space_id = updated.space_id
    else:
        payload = {
            "title": DISPLAY_NAME,
            "description": DESCRIPTION,
            "warehouse_id": warehouse_id,
            "serialized_space": serialized,
        }
        updated = client.api_client.do("PATCH", f"/api/2.0/genie/spaces/{existing.space_id}", body=payload)
        space_id = updated.get("space_id") or existing.space_id
    print(f"✓ Updated Genie Space: {space_id}")
else:
    # Create new space
    if hasattr(client.genie, "create_space"):
        created = client.genie.create_space(
            title=DISPLAY_NAME,
            description=DESCRIPTION,
            warehouse_id=warehouse_id,
            serialized_space=serialized,
        )
        space_id = created.space_id
    else:
        payload = {
            "title": DISPLAY_NAME,
            "description": DESCRIPTION,
            "warehouse_id": warehouse_id,
            "serialized_space": serialized,
        }
        created = client.api_client.do("POST", "/api/2.0/genie/spaces", body=payload)
        space_id = created.get("space_id")
    print(f"✓ Created Genie Space: {space_id}")

print()
print(f"DATABRICKS_GENIE_SPACE_ID={space_id}")
print()
print("Add this value to your databricks.yml targets.dev.variables section.")

# Grant the deploying user CAN_MANAGE so they can update/delete the space later
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

try:
    current_user = client.current_user.me()
    client.permissions.update(
        request_object_type="genie",
        request_object_id=space_id,
        access_control_list=[
            AccessControlRequest(
                user_name=current_user.user_name,
                permission_level=PermissionLevel.CAN_MANAGE,
            )
        ],
    )
    print(f"✓ Granted CAN_MANAGE to {current_user.user_name}")
except Exception as e:
    print(f"⚠ Could not grant CAN_MANAGE (non-fatal): {e}")